# Vietnamese License Plate Recognition — CCT-XS Fine-Tuning Pipeline
This notebook fine-tunes a Compact Convolutional Transformer (CCT-XS) model for Vietnamese license plate recognition using `fast-plate-ocr` on Kaggle GPU.

### Operational Safety & Constraints:
- **DRY_RUN mode**: Defaults to `DRY_RUN = True` for safe static and CI validation (0 training epochs, 0 weight updates).
- **Paths**: Uses `/kaggle/input/` for read-only datasets and `/kaggle/working/` for outputs and checkpoints.
- **Air-gapped compatibility**: Installs dependencies offline from `/kaggle/input/plate-training-wheels` when internet is disabled.
- **Privacy & Decree 13/2023/ND-CP**: No real plate images or raw secrets are printed or committed.

In [ ]:
# Cell 1: Environment & Execution Mode Configuration
import os
import sys
from pathlib import Path

# Execution switch: Set DRY_RUN = False ONLY when running real GPU training on Kaggle
DRY_RUN: bool = True

# Pre-set backend before any Keras or Torch imports
os.environ["KERAS_BACKEND"] = "torch"

# Resolve repository root and source paths for local and Kaggle execution
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "training" / "src").is_dir():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "src").is_dir():
    REPO_ROOT = NOTEBOOK_DIR.parent.parent
else:
    REPO_ROOT = Path("/kaggle/working")

SRC_DIR = REPO_ROOT / "training" / "src"
if SRC_DIR.is_dir() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
SCRIPTS_DIR = REPO_ROOT / "training" / "scripts"
if SCRIPTS_DIR.is_dir() and str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Standard Kaggle paths with local fallbacks
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

if not KAGGLE_INPUT_DIR.exists():
    KAGGLE_INPUT_DIR = REPO_ROOT / "training" / "data"
if not KAGGLE_WORKING_DIR.exists():
    KAGGLE_WORKING_DIR = REPO_ROOT / "training" / "outputs"
    KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = KAGGLE_WORKING_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = KAGGLE_WORKING_DIR / "exported_models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] DRY_RUN = {DRY_RUN}")
print(f"[CONFIG] Backend: {os.environ['KERAS_BACKEND']}")
print(f"[CONFIG] Input directory: {KAGGLE_INPUT_DIR}")
print(f"[CONFIG] Working directory: {KAGGLE_WORKING_DIR}")
print(f"[CONFIG] Checkpoint directory: {CHECKPOINT_DIR}")
print(f"[CONFIG] Export directory: {EXPORT_DIR}")

In [ ]:
# Cell 2: Offline Wheel Installation (Air-Gapped Environment)
import subprocess
import sys

WHEELS_DATASET_DIR = KAGGLE_INPUT_DIR / "plate-training-wheels"

if WHEELS_DATASET_DIR.exists():
    print(f"[SETUP] Installing offline wheels from {WHEELS_DATASET_DIR}...")
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        f"--find-links={WHEELS_DATASET_DIR}",
        "fast-plate-ocr",
        "keras",
        "albumentations",
        "onnxslim",
    ]
    res = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(f"[SETUP] Pip install returncode: {res.returncode}")
else:
    print(
        "[SETUP] No offline wheel directory found. Relying on pre-installed environment packages."
    )

In [ ]:
# Cell 3: Backend & Random Seed Setup for Reproducibility
import os
import random

import keras
import numpy as np

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

print(f"[FRAMEWORK] Keras version: {keras.__version__}")
print(f"[FRAMEWORK] Active Backend: {keras.backend.backend()}")
print(f"[FRAMEWORK] Deterministic random seed configured: {RANDOM_SEED}")

In [ ]:
# Cell 4: Config Loading & Schema Validation
from pathlib import Path

from fast_plate_ocr.train.model.config import PlateConfig, load_plate_config_from_yaml
from fast_plate_ocr.train.model.model_schema import (
    CCTModelConfig,
    load_model_config_from_yaml,
)

from plate_training.config import TrainConfig

# Resolve config locations across Kaggle and local paths
candidate_dirs = [
    REPO_ROOT / "training" / "configs",
    Path("training/configs"),
    Path("configs"),
    KAGGLE_INPUT_DIR / "vietnamese-license-plate-manifest",
]


def find_config_file(filename: str) -> Path:
    for c_dir in candidate_dirs:
        candidate = c_dir / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Configuration file {filename} not found in candidates: {candidate_dirs}"
    )


PLATE_CONFIG_PATH = find_config_file("plate_config_vn.yaml")
MODEL_CONFIG_PATH = find_config_file("model_cct_xs.yaml")
TRAIN_CONFIG_PATH = find_config_file("train_cct_xs_baseline.yaml")

# Load and validate with real Pydantic schemas (extra='forbid')
plate_cfg: PlateConfig = load_plate_config_from_yaml(PLATE_CONFIG_PATH)
model_cfg: CCTModelConfig = load_model_config_from_yaml(MODEL_CONFIG_PATH)
train_cfg: TrainConfig = TrainConfig.from_yaml(TRAIN_CONFIG_PATH)

# Verify key architectural invariants
assert plate_cfg.max_plate_slots == 10, (
    f"Expected 10 plate slots, got {plate_cfg.max_plate_slots}"
)
assert len(plate_cfg.alphabet) == 37, (
    f"Expected alphabet length 37, got {len(plate_cfg.alphabet)}"
)
assert plate_cfg.img_height == 64 and plate_cfg.img_width == 128
assert model_cfg.model == "cct"
assert model_cfg.transformer_encoder.projection_dim == 96
assert train_cfg.seed == 42
assert train_cfg.training.batch_size == 64

print(
    f"[CONFIG] PlateConfig validated: slots={plate_cfg.max_plate_slots}, alphabet_len={len(plate_cfg.alphabet)}"
)
print(
    f"[CONFIG] ModelConfig validated: model={model_cfg.model}, proj_dim={model_cfg.transformer_encoder.projection_dim}"
)
print(
    f"[CONFIG] TrainConfig validated: seed={train_cfg.seed}, batch_size={train_cfg.training.batch_size}, epochs={train_cfg.training.epochs}"
)

In [ ]:
# Cell 5: Dataset Manifest Verification & Split Leakage Check
import hashlib
import json
from pathlib import Path

import cv2
import numpy as np

from plate_training.manifest import ManifestItem, load_manifest, save_manifest
from plate_training.split import DataLeakageError, validate_splits

# 1. Load Frozen Test Registry (hashes only, zero plate text, zero image bytes)
registry_candidates = [
    REPO_ROOT / "training" / "data" / "frozen_test_registry.json",
    Path("training/data/frozen_test_registry.json"),
    Path("data/frozen_test_registry.json"),
    KAGGLE_INPUT_DIR
    / "vietnamese-license-plate-manifest"
    / "frozen_test_registry.json",
]
registry_path = next((p for p in registry_candidates if p.is_file()), None)
if registry_path is None:
    raise FileNotFoundError(
        "frozen_test_registry.json required for leakage verification."
    )

with open(registry_path, encoding="utf-8") as f:
    registry_data = json.load(f)

assert len(registry_data) == 322, (
    f"Expected 322 frozen test entries, got {len(registry_data)}"
)
frozen_sha256 = {r["sha256"].lower() for r in registry_data}

print(
    f"[LEAKAGE] Loaded frozen test registry: {len(registry_data)} items ({len(frozen_sha256)} unique SHA-256)."
)

# 2. Synthetic Guardrail Verification (Demonstrates catching leakage via validate_splits)
synthetic_leaked_sha = next(iter(frozen_sha256))
caught_leakage = False
try:
    leaked_item = ManifestItem(
        image_path="leak_test.png",
        sha256=synthetic_leaked_sha,
        plate_text="29A12345",
        split="train",
        provenance="synthetic-offline-harness",
        license="CC0",
        group_id="leak_group_001",
    )
    validate_splits(
        [leaked_item],
        registry_path=registry_path,
        allow_missing_files=True,
        check_phash=False,
    )
except DataLeakageError:
    caught_leakage = True
assert caught_leakage, "Fail-closed check failed to catch exact SHA-256 leakage!"

# 3. Resolve active dataset manifest (Live Kaggle dataset or Offline Synthetic Fallback)
real_manifest_path = (
    KAGGLE_INPUT_DIR / "vietnamese-license-plate-manifest" / "manifest.json"
)
if real_manifest_path.is_file():
    real_items = load_manifest(real_manifest_path)
    validate_splits(
        real_items,
        base_dir=real_manifest_path.parent,
        registry_path=registry_path,
        allow_missing_files=False,
    )
    active_manifest_path = real_manifest_path
    manifest_base_dir = real_manifest_path.parent
    print(
        f"[DATA] Real manifest verified: {len(real_items)} items disjoint from frozen test registry."
    )
else:
    print(
        "[DATA] No live manifest in environment. Constructing verified synthetic dataset for offline pipeline execution..."
    )
    synth_dir = KAGGLE_WORKING_DIR / "synthetic_dataset"
    synth_dir.mkdir(parents=True, exist_ok=True)
    active_manifest_path = synth_dir / "manifest.json"
    manifest_base_dir = synth_dir

    synth_items: list[ManifestItem] = []
    rng_sample = np.random.default_rng(RANDOM_SEED)
    sample_specs = [
        ("synth_train_0.png", "29A12345", "train", "synth_veh_001"),
        ("synth_train_1.png", "51G99999", "train", "synth_veh_002"),
        ("synth_val_0.png", "43B88888", "val", "synth_veh_003"),
        ("synth_val_1.png", "30E77777", "val", "synth_veh_004"),
    ]
    for filename, plate_text, split_name, g_id in sample_specs:
        img_path = synth_dir / filename
        crop_data = rng_sample.integers(60, 200, size=(64, 128, 3), dtype=np.uint8)
        cv2.imwrite(str(img_path), crop_data)
        file_sha = hashlib.sha256(img_path.read_bytes()).hexdigest()
        synth_items.append(
            ManifestItem(
                image_path=filename,
                sha256=file_sha,
                plate_text=plate_text,
                split=split_name,
                provenance="synthetic-offline-harness",
                license="CC0",
                group_id=g_id,
            )
        )
    save_manifest(synth_items, active_manifest_path)
    validate_splits(
        synth_items,
        base_dir=synth_dir,
        registry_path=registry_path,
        allow_missing_files=False,
    )
    print(
        f"[DATA] Synthetic dataset generated and verified via validate_splits: {len(synth_items)} items at {active_manifest_path}"
    )

In [ ]:
# Cell 6: Model Architecture Construction & Pretrained Weights Loading
import os
from pathlib import Path

import numpy as np

try:
    from export_model import patch_patch_extractor_if_needed
except ImportError:
    from training.scripts.export_model import patch_patch_extractor_if_needed

from plate_training.model import build_model, count_model_parameters, load_aligned_model

# 1. Apply TorchScript tracing compatibility patch
patch_patch_extractor_if_needed()

# 2. Discover and resolve pretrained cct_xs_v2_global.keras weights
# Candidate paths strictly: (1) PLATE_PRETRAINED_WEIGHTS, (2) /kaggle/input/..., (3) PLATE_TRAINING_CACHE
candidate_weights = [
    Path(os.environ["PLATE_PRETRAINED_WEIGHTS"])
    if os.environ.get("PLATE_PRETRAINED_WEIGHTS")
    else None,
    KAGGLE_INPUT_DIR / "cct-xs-v2-global-weights" / "cct_xs_v2_global.keras",
    KAGGLE_INPUT_DIR / "cct_xs_v2_global.keras",
    Path(
        os.environ.get(
            "PLATE_TRAINING_CACHE", Path.home() / ".cache" / "plate_training"
        )
    )
    / "cct_xs_v2_global.keras",
]
weights_path = next(
    (p for p in candidate_weights if p is not None and p.is_file()), None
)

# 3. Build aligned CCT-XS model and load pretrained weights with tensor verification
if weights_path is not None:
    print(
        f"[MODEL] Loading pretrained weights from {weights_path} with post-load tensor verification..."
    )
    model, v_info = load_aligned_model(
        weights_path=weights_path,
        model_config_path=MODEL_CONFIG_PATH
        if "MODEL_CONFIG_PATH" in globals()
        else None,
        plate_config_path=PLATE_CONFIG_PATH
        if "PLATE_CONFIG_PATH" in globals()
        else None,
        drop_region_head=True,
        verify_tensors=True,
        return_verification_info=True,
    )
    print(
        f"[MODEL] Pretrained weights verified: {v_info['compared_tensors_count']} plate tensors "
        f"(max diff: {v_info['max_weight_diff']:.6e}), skipped region layers: {v_info['skipped_region_head_layers']}"
    )
else:
    print(
        "[MODEL] Pretrained weights file not found in candidate paths. "
        "Proceeding with initialized weights (expected in offline test harnesses)."
    )
    model = build_model(
        model_cfg=model_cfg,
        plate_cfg=plate_cfg,
        enable_region_head=False,
    )

param_info = count_model_parameters(model)
print(
    f"[MODEL] CCT-XS Model ready. Total params: {param_info['total']:,} (Trainable: {param_info['trainable']:,})"
)

# 4. Run 1 real forward pass on synthetic batch (uint8, NHWC)
rng = np.random.default_rng(RANDOM_SEED)
synthetic_batch = rng.integers(0, 256, size=(2, 64, 128, 3), dtype=np.uint8)
raw_output = model(synthetic_batch, training=False)

plate_logits = raw_output["plate"] if isinstance(raw_output, dict) else raw_output
assert plate_logits.shape == (
    2,
    10,
    37,
), f"Unexpected output shape: {plate_logits.shape}"
assert "float32" in str(plate_logits.dtype).lower(), (
    f"Unexpected output dtype: {plate_logits.dtype}"
)

print(
    f"[FORWARD_PASS] 1 real forward pass succeeded. Output tensor shape: {plate_logits.shape}, dtype: {plate_logits.dtype}"
)

In [ ]:
# Cell 7: Dataset Construction, Model Compilation & Training Loop Switch
import time

import keras
from fast_plate_ocr.train.model.loss import cce_loss
from fast_plate_ocr.train.model.metric import cat_acc_metric, plate_acc_metric

from plate_training.checkpoint import find_latest_checkpoint
from plate_training.dataset import build_dataset


# 1. Benchmark Callback measuring exact wall-clock seconds per epoch
class EpochTimingCallback(keras.callbacks.Callback):
    """Measures exact wall-clock seconds per epoch and logs throughput."""

    def on_train_begin(self, logs=None):
        self.epoch_times = []
        self.train_start_time = time.perf_counter()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.perf_counter() - self.epoch_start
        self.epoch_times.append(elapsed)
        print(f"[BENCHMARK] Epoch {epoch + 1}: {elapsed:.3f} seconds")

    def on_train_end(self, logs=None):
        total_time = time.perf_counter() - self.train_start_time
        avg_epoch = sum(self.epoch_times) / max(len(self.epoch_times), 1)
        print(
            f"[BENCHMARK] Total Training Time: {total_time:.2f}s | "
            f"Mean Epoch Duration: {avg_epoch:.3f}s"
        )


# 2. Build training and validation datasets from verified manifest
train_dataset = build_dataset(
    manifest_path=active_manifest_path,
    split="train",
    batch_size=train_cfg.training.batch_size,
    shuffle=True,
    seed=train_cfg.seed,
    base_dir=manifest_base_dir,
    plate_config_path=PLATE_CONFIG_PATH,
    drop_region_head=True,
    validate_split=True,
)

val_dataset = build_dataset(
    manifest_path=active_manifest_path,
    split="val",
    batch_size=train_cfg.training.batch_size,
    shuffle=False,
    seed=train_cfg.seed,
    base_dir=manifest_base_dir,
    plate_config_path=PLATE_CONFIG_PATH,
    drop_region_head=True,
    validate_split=True,
)

print(
    f"[DATASET] Datasets built: train_batches={len(train_dataset)}, val_batches={len(val_dataset)}"
)

# 3. Model Compilation following upstream fast-plate-ocr standards
total_steps = train_cfg.training.epochs * len(train_dataset)
warmup_steps = int(train_cfg.optimizer.warmup_fraction * total_steps)

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0 if warmup_steps > 0 else train_cfg.optimizer.lr,
    decay_steps=max(1, total_steps - warmup_steps),
    alpha=train_cfg.optimizer.final_lr_factor,
    warmup_steps=warmup_steps,
    warmup_target=train_cfg.optimizer.lr if warmup_steps > 0 else None,
)

optimizer = keras.optimizers.AdamW(
    learning_rate=cosine_decay,
    weight_decay=train_cfg.optimizer.weight_decay,
    clipnorm=train_cfg.optimizer.clipnorm,
    use_ema=train_cfg.optimizer.use_ema,
)
if hasattr(optimizer, "exclude_from_weight_decay") and train_cfg.optimizer.wd_ignore:
    optimizer.exclude_from_weight_decay(
        var_names=[
            name.strip()
            for name in train_cfg.optimizer.wd_ignore.split(",")
            if name.strip()
        ]
    )

loss_fn = cce_loss(
    vocabulary_size=plate_cfg.vocabulary_size,
    label_smoothing=train_cfg.loss.label_smoothing,
)

p_acc = plate_acc_metric(
    max_plate_slots=plate_cfg.max_plate_slots,
    vocabulary_size=plate_cfg.vocabulary_size,
)
c_acc = cat_acc_metric(
    max_plate_slots=plate_cfg.max_plate_slots,
    vocabulary_size=plate_cfg.vocabulary_size,
)

model.compile(
    optimizer=optimizer,
    loss={"plate": loss_fn},
    metrics={"plate": [p_acc, c_acc]},
    jit_compile=False,
)
print(
    "[COMPILE] Model compiled with AdamW, CosineDecay, CCE loss, and accuracy metrics."
)

# 4. Checkpoint discovery & resume
latest_ckpt, initial_epoch = find_latest_checkpoint(CHECKPOINT_DIR)
print(
    f"[CHECKPOINT] Discovery result: latest={latest_ckpt}, initial_epoch={initial_epoch}"
)

if latest_ckpt and latest_ckpt.is_file():
    print(f"[CHECKPOINT] Resuming from checkpoint weights: {latest_ckpt}...")
    model.load_weights(str(latest_ckpt))

# 5. Training Callbacks Configuration
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / "cct_xs_epoch_{epoch:03d}.keras"),
        save_best_only=train_cfg.checkpoint.save_best_only,
        monitor="val_plate_acc",
        mode="max",
        save_weights_only=train_cfg.checkpoint.save_weights_only,
    ),
    keras.callbacks.EarlyStopping(
        patience=train_cfg.early_stopping.patience,
        monitor=train_cfg.early_stopping.metric,
        mode=train_cfg.early_stopping.mode,
        restore_best_weights=True,
    ),
    EpochTimingCallback(),
]

# 6. Real Training Execution or Zero-Training DRY_RUN Evaluation
if not DRY_RUN:
    print("[TRAIN] Starting full model training on Kaggle GPU...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=train_cfg.training.epochs,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
    )
else:
    print(
        "[DRY_RUN] DRY_RUN is True. Zero epochs executed, zero weight updates on this machine."
    )
    print(
        f"[DRY_RUN] Checkpoint discovery and resume logic verified (epoch: {initial_epoch})."
    )
    eval_metrics = model.evaluate(val_dataset, verbose=1)
    print(f"[DRY_RUN] Validation evaluation completed: {eval_metrics}")

In [ ]:
# Cell 8: Validation Evaluation via Standard plate_training.metrics
from plate_training.metrics import (
    compute_cer,
    compute_character_accuracy,
    compute_exact_match,
)

# Fetch first batch from real/synthetic validation dataset
val_batch_x, val_batch_y = val_dataset[0]
val_raw = model(val_batch_x, training=False)
val_logits = val_raw["plate"] if isinstance(val_raw, dict) else val_raw

if hasattr(val_logits, "detach"):
    logits_np = val_logits.detach().cpu().numpy()
else:
    logits_np = np.asarray(val_logits)

# Decode predictions using plate alphabet
pred_indices = np.argmax(logits_np, axis=-1)
alphabet = plate_cfg.alphabet
decoded_preds = [
    "".join(alphabet[idx] for idx in sample).rstrip(plate_cfg.pad_char)
    for sample in pred_indices
]

# Ground truths from validation batch
y_plate_true = val_batch_y["plate"]
gt_indices = np.argmax(y_plate_true, axis=-1)
ground_truths = [
    "".join(alphabet[idx] for idx in sample).rstrip(plate_cfg.pad_char)
    for sample in gt_indices
]

exact_match_ratio = compute_exact_match(ground_truths, decoded_preds)
char_accuracy_ratio = compute_character_accuracy(ground_truths, decoded_preds)
cer_ratio = compute_cer(ground_truths, decoded_preds)

metrics_summary = {
    "evaluated_samples": len(decoded_preds),
    "exact_match": exact_match_ratio,
    "character_accuracy": char_accuracy_ratio,
    "cer": cer_ratio,
}
print(f"[METRICS] Validation evaluation computed dynamically: {metrics_summary}")

In [ ]:
# Cell 9: ONNX Export, Parity Check & Dynamic run_summary.json
import json
import subprocess

try:
    from export_model import export_keras_to_onnx, patch_patch_extractor_if_needed
except ImportError:
    from training.scripts.export_model import (
        export_keras_to_onnx,
        patch_patch_extractor_if_needed,
    )
from plate_training.manifest import compute_file_sha256
from plate_training.onnx_parity import check_onnx_parity

patch_patch_extractor_if_needed()
output_onnx_path = EXPORT_DIR / "cct_xs_v2_global.onnx"

# 1. Export Keras model to ONNX adhering strictly to ADR-0007
exported_path = export_keras_to_onnx(
    model=model,
    plate_cfg=plate_cfg,
    output_onnx_path=output_onnx_path,
    simplify=True,
)
assert exported_path.is_file(), f"Exported ONNX file missing: {exported_path}"

# 2. Real numerical parity verification between Keras and ONNX Runtime
parity_report = check_onnx_parity(
    keras_model_or_predict=model,
    onnx_model_or_session=exported_path,
    input_shape=(plate_cfg.img_height, plate_cfg.img_width, plate_cfg.num_channels),
    batch_size=4,
    atol=1e-4,
    rtol=1e-4,
    raise_on_error=True,
)

# 3. Dynamic artifact checksums & telemetry (zero hardcoded values)
onnx_sha = compute_file_sha256(exported_path)
onnx_size = exported_path.stat().st_size

try:
    git_rev = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_rev = "unknown"

run_summary = {
    "dry_run": DRY_RUN,
    "git_commit": git_rev,
    "frameworks": {
        "keras": keras.__version__,
        "backend": keras.backend.backend(),
    },
    "model": {
        "architecture": model_cfg.model,
        "parameters": model.count_params(),
        "input_shape": [
            None,
            plate_cfg.img_height,
            plate_cfg.img_width,
            plate_cfg.num_channels,
        ],
        "output_shape": [None, plate_cfg.max_plate_slots, len(plate_cfg.alphabet)],
    },
    "artifacts": {
        "onnx_path": str(exported_path),
        "onnx_size_bytes": onnx_size,
        "onnx_sha256": onnx_sha,
    },
    "parity": {
        "passed": parity_report["passed"],
        "max_abs_diff": parity_report["max_abs_diff"],
        "mean_abs_diff": parity_report["mean_abs_diff"],
    },
    "validation_metrics": metrics_summary,
}

summary_path = EXPORT_DIR / "run_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

print(f"[EXPORT] Run summary saved to {summary_path}")
print(f"[EXPORT] ONNX SHA-256: {onnx_sha} ({onnx_size} bytes)")
print(
    f"[EXPORT] Numerical parity: passed={parity_report['passed']}, max_abs_diff={parity_report['max_abs_diff']:.6e}"
)

## Summary & Audit Checklist
- [x] Valid nbformat v4 specification.
- [x] 0 stored outputs in git repository.
- [x] 0 secrets or API keys hardcoded.
- [x] Paths adhere to `/kaggle/input/` and `/kaggle/working/`.
- [x] `DRY_RUN = True` switch guards against local weight updates.